In [0]:
# Databricks notebook source
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

print("=" * 60)
print("FASE 4: ANÁLISE & VISUALIZAÇÃO")
print("=" * 60)

VOLUME_PATH = "/Volumes/workspace/default/nyc_taxi"
PROCESSED_PATH = f"{VOLUME_PATH}/processed"
ANALYTICS_PATH = f"{VOLUME_PATH}/analytics/aggregations"

# Carregar dados já transformados (Fase 3)
df = spark.read.format("delta").load(f"{PROCESSED_PATH}/featured/taxi_featured")
df.createOrReplaceTempView("taxi_featured")

print(f"\n📊 Linhas carregadas: {df.count():,}")
print(f"📋 Colunas disponíveis: {len(df.columns)}")

In [0]:
# TAREFA 4.2: Análises por Período
print("\n" + "="*60)
print("TAREFA 4.2: Comparação por Ano/Mês")
print("="*60)

por_periodo = spark.sql("""
SELECT 
    pickup_year,
    pickup_month,
    COUNT(*) as total_viagens,
    ROUND(SUM(fare_amount), 0) as receita_total,
    ROUND(AVG(fare_amount), 2) as tarifa_media,
    ROUND(AVG(trip_distance_km), 2) as distancia_media_km,
    ROUND(AVG(CASE WHEN payment_type = 1 THEN tip_percentage END), 1) as gorjeta_media_pct
FROM taxi_featured
GROUP BY pickup_year, pickup_month
ORDER BY pickup_year, pickup_month
""")
display(por_periodo)

In [0]:
# TAREFA 4.3: Padrões Horários
print("\n" + "="*60)
print("TAREFA 4.3: Padrões por Hora do Dia")
print("="*60)

hourly_trends = spark.sql("""
SELECT 
    pickup_hour,
    COUNT(*) as total_viagens,
    ROUND(AVG(fare_amount), 2) as tarifa_media,
    ROUND(AVG(CASE WHEN payment_type = 1 THEN tip_percentage END), 1) as gorjeta_media_pct,
    ROUND(AVG(trip_distance_km), 2) as distancia_media_km,
    ROUND(AVG(CASE WHEN duracao_valida = 1 THEN speed_kmh END), 2) as velocidade_media_kmh
FROM taxi_featured
GROUP BY pickup_hour
ORDER BY pickup_hour
""")
display(hourly_trends)

# Salvar para dashboard
hourly_trends.write.format("delta").mode("overwrite") \
    .save(f"{ANALYTICS_PATH}/hourly_trends")
print("\n✅ Agregação salva: hourly_trends")

In [0]:
# TAREFA 4.4: Padrões por Dia da Semana
print("\n" + "="*60)
print("TAREFA 4.4: Padrões por Dia da Semana")
print("="*60)

daily_trends = spark.sql("""
SELECT 
    pickup_day_of_week,
    CASE 
        WHEN pickup_day_of_week = 1 THEN 'Domingo'
        WHEN pickup_day_of_week = 2 THEN 'Segunda'
        WHEN pickup_day_of_week = 3 THEN 'Terça'
        WHEN pickup_day_of_week = 4 THEN 'Quarta'
        WHEN pickup_day_of_week = 5 THEN 'Quinta'
        WHEN pickup_day_of_week = 6 THEN 'Sexta'
        ELSE 'Sábado'
    END as dia_semana,
    is_weekend,
    COUNT(*) as total_viagens,
    ROUND(AVG(fare_amount), 2) as tarifa_media,
    ROUND(AVG(CASE WHEN payment_type = 1 THEN tip_percentage END), 1) as gorjeta_media_pct
FROM taxi_featured
GROUP BY pickup_day_of_week, is_weekend
ORDER BY pickup_day_of_week
""")
display(daily_trends)

daily_trends.write.format("delta").mode("overwrite") \
    .save(f"{ANALYTICS_PATH}/daily_trends")
print("\n✅ Agregação salva: daily_trends")

In [0]:
# TAREFA 4.5: Top Zonas de Pickup
print("\n" + "="*60)
print("TAREFA 4.5: Top 20 Zonas de Pickup")
print("="*60)

top_zonas = spark.sql("""
SELECT 
    ROUND(pickup_latitude, 2) as pickup_lat,
    ROUND(pickup_longitude, 2) as pickup_lng,
    COUNT(*) as total_viagens,
    ROUND(AVG(fare_amount), 2) as tarifa_media,
    ROUND(AVG(trip_distance_km), 2) as distancia_media_km
FROM taxi_featured
WHERE pickup_latitude IS NOT NULL 
  AND pickup_longitude IS NOT NULL
GROUP BY ROUND(pickup_latitude, 2), ROUND(pickup_longitude, 2)
ORDER BY total_viagens DESC
LIMIT 20
""")
display(top_zonas)

top_zonas.write.format("delta").mode("overwrite") \
    .save(f"{ANALYTICS_PATH}/top_zonas")
print("\n✅ Agregação salva: top_zonas")

In [0]:
# TAREFA 4.5b: Análise Detalhada de Aeroportos
print("\n" + "="*60)
print("TAREFA 4.5b: Viagens de Aeroporto por Hora")
print("="*60)

aeroporto_hora = spark.sql("""
SELECT 
    airport_type,
    pickup_hour,
    COUNT(*) as total_viagens,
    ROUND(AVG(fare_amount), 2) as tarifa_media
FROM taxi_featured
WHERE is_airport_trip = 1
GROUP BY airport_type, pickup_hour
ORDER BY airport_type, pickup_hour
""")
display(aeroporto_hora)

In [0]:
# TAREFA 4.6: Gorjeta por Segmento
print("\n" + "="*60)
print("TAREFA 4.6: Fatores que Influenciam Gorjeta")
print("="*60)

gorjeta_segmento = spark.sql("""
SELECT 
    tip_category,
    is_weekend,
    is_rush_hour,
    is_airport_trip,
    COUNT(*) as total_viagens,
    ROUND(AVG(fare_amount), 2) as tarifa_media
FROM taxi_featured
WHERE payment_type = 1
GROUP BY tip_category, is_weekend, is_rush_hour, is_airport_trip
ORDER BY total_viagens DESC
""")
display(gorjeta_segmento)

gorjeta_segmento.write.format("delta").mode("overwrite") \
    .save(f"{ANALYTICS_PATH}/gorjeta_segmento")
print("\n✅ Agregação salva: gorjeta_segmento")

In [0]:
# TAREFA 4.7: Análise por Tipo de Pagamento
print("\n" + "="*60)
print("TAREFA 4.7: Tipo de Pagamento")
print("="*60)

payment_analysis = spark.sql("""
SELECT 
    payment_type,
    COUNT(*) as total_viagens,
    ROUND(SUM(fare_amount), 0) as receita_total,
    ROUND(AVG(fare_amount), 2) as tarifa_media,
    ROUND(AVG(CASE WHEN payment_type = 1 THEN tip_amount END), 2) as gorjeta_media
FROM taxi_featured
GROUP BY payment_type
ORDER BY total_viagens DESC
""")
display(payment_analysis)

payment_analysis.write.format("delta").mode("overwrite") \
    .save(f"{ANALYTICS_PATH}/payment_analysis")
print("\n✅ Agregação salva: payment_analysis")

In [0]:
# TAREFA 4.8: Visualização - Padrão Horário
display(hourly_trends)
# 💡 Clique no ícone de gráfico abaixo da tabela e escolha "Line Chart"
# eixo X: pickup_hour, eixo Y: total_viagens

In [0]:
# TAREFA 4.8b: Comparação Aeroporto vs Standard
comparacao_tipo = spark.sql("""
SELECT 
    CASE WHEN is_airport_trip = 1 THEN airport_type ELSE 'Standard' END as tipo_viagem,
    COUNT(*) as total_viagens,
    ROUND(AVG(fare_amount), 2) as tarifa_media,
    ROUND(AVG(trip_distance_km), 2) as distancia_media_km,
    ROUND(AVG(CASE WHEN duracao_valida = 1 THEN trip_duration_minutes END), 1) as duracao_media_min
FROM taxi_featured
GROUP BY CASE WHEN is_airport_trip = 1 THEN airport_type ELSE 'Standard' END
ORDER BY total_viagens DESC
""")
display(comparacao_tipo)

## 📋 Resumo — Fase 4: Análise & Visualização

### Agregações Criadas (salvas em Delta Lake para dashboard)
- hourly_trends (24 linhas) — padrões por hora do dia
- daily_trends (7 linhas) — padrões por dia da semana
- top_zonas (20 linhas) — top zonas de pickup
- payment_analysis (5 linhas) — análise por tipo de pagamento
- gorjeta_segmento (32 linhas) — gorjeta por dia/hora/aeroporto

### Validação Cruzada
- Soma de tip_category (Fase 3) = total de payment_type=1 (Fase 4):
  30.751.907 viagens em ambos os cálculos — consistência confirmada

### Confirmação do Achado da Fase 2
- Após a limpeza da Fase 3, a diferença de distância média entre
  2015-01 (4,51 km) e 2016-01 (4,70 km) ficou muito menor,
  confirmando que os outliers extremos eram a causa da discrepância
  original observada na Fase 2 (13,55 vs 4,68 milhas)

### Gorjeta por Dia da Semana (com filtro correto de payment_type=1)
- Maior gorjeta média: Domingo (25,4%)
- Menor gorjeta média: Sábado (20,8%)
> Nota: o achado da Fase 2 sobre "sexta ter maior gorjeta" foi calculado
> sem filtrar payment_type, portanto o resultado atual é mais confiável

### Aeroportos: Estabilidade da Tarifa Fixa
- JFK: tarifa entre $51,70 e $51,99 ao longo de todas as 24 horas do dia
  (variação mínima), consistente com o conceito de tarifa fixa (flat rate)

### Comparação Standard vs Aeroporto
| Tipo | Viagens | Tarifa Média | Distância Média | Duração Média |
|---|---|---|---|---|
| Standard | 45.926.683 | $11,48 | 4,15 km | 12,5 min |
| JFK | 885.874 | $51,98 | 28,84 km | 43,3 min |
| Newark | 69.593 | $66,32 | 27,59 km | 36,1 min |

### Padrões Gerais
- Pico de viagens: 18h-19h (~2,4M viagens/hora)
- Menor volume: madrugada (3h-4h, ~500-680k viagens/hora)
- Sábado tem o maior volume de viagens totais, mas a menor gorjeta média